In [1]:
import seaborn as sns
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder as OHE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay,
    f1_score, recall_score, roc_auc_score
)

In [3]:
trainM = pd.read_csv('../../../../Data/fraudTrain_model.csv')
trainM.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 27 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   trans_date_trans_time   1296675 non-null  object 
 1   cc_num                  1296675 non-null  int64  
 2   category                1296675 non-null  object 
 3   amt                     1296675 non-null  float64
 4   gender                  1296675 non-null  object 
 5   city                    1296675 non-null  object 
 6   state                   1296675 non-null  object 
 7   zip                     1296675 non-null  int64  
 8   job                     1296675 non-null  object 
 9   is_fraud                1296675 non-null  int64  
 10  txn_distance_km         1296675 non-null  float64
 11  year                    1296675 non-null  int64  
 12  month                   1296675 non-null  int64  
 13  is_weekend              1296675 non-null  int64  
 14  is

We grouped transactions by cc_num and used ***diff()*** to calculate the time gap between consecutive transactions. Because each credit card’s first transaction has no prior record, this results in one missing value per card. With ***983 unique credit cards*** in the dataset, we therefore observe ***983 NaN values***.

During the modeling stage, these missing values will be handled using median imputation computed from the training fold and applied consistently to the validation set to prevent data leakage.

### Time-series Target Encoding

time-based target encoding
TimeSeriesSplit
XGBoost
Precision@K / Recall@K

In [4]:
trainM['category'].unique()

array(['misc_net', 'gas_transport', 'kids_pets', 'home', 'shopping_net',
       'food_dining', 'personal_care', 'grocery_pos', 'entertainment',
       'shopping_pos', 'misc_pos', 'travel', 'health_fitness',
       'grocery_net'], dtype=object)

In [5]:
[trainM['trans_date_trans_time'].min(), trainM['trans_date_trans_time'].max()]

['2019-01-01 00:00:18', '2020-06-21 12:13:37']

In [6]:
from sklearn.model_selection import TimeSeriesSplit

trainM = trainM.sort_values('trans_date_trans_time').reset_index(drop=True) # 这里需要写level=0, level=0代表着列，等于1代表着行？
tscv = TimeSeriesSplit(n_splits=5) # 什么情况下需要写 test_size=2

for i, (train_idx, val_idx) in enumerate(tscv.split(trainM), 1): # 这里的1是什么意思
    train_fold = trainM.iloc[train_idx]
    val_fold = trainM.iloc[val_idx]

    print(f"Fold {i}:")
    print(train_fold['trans_date_trans_time'].min(), "→", train_fold['trans_date_trans_time'].max())
    print(val_fold['trans_date_trans_time'].min(), "→", val_fold['trans_date_trans_time'].max())

    print("-"*50)

Fold 1:
2019-01-01 00:00:18 → 2019-04-20 12:53:56
2019-04-20 12:54:06 → 2019-07-12 23:50:14
--------------------------------------------------
Fold 2:
2019-01-01 00:00:18 → 2019-07-12 23:50:14
2019-07-12 23:50:15 → 2019-10-03 07:36:02
--------------------------------------------------
Fold 3:
2019-01-01 00:00:18 → 2019-10-03 07:36:02
2019-10-03 07:36:54 → 2019-12-18 17:07:05
--------------------------------------------------
Fold 4:
2019-01-01 00:00:18 → 2019-12-18 17:07:05
2019-12-18 17:07:56 → 2020-03-24 15:14:30
--------------------------------------------------
Fold 5:
2019-01-01 00:00:18 → 2020-03-24 15:14:30
2020-03-24 15:15:08 → 2020-06-21 12:13:37
--------------------------------------------------


Time series sort → folding on feature engineering → modeling → model evaluation → performance summary by model (train.csv)

In [7]:
def time_based_target_encode_train_val(train_fold, val_fold, col, target):
    """
    Create time-safe target encoding for a categorical column.

    For the training fold:
    - Each row only uses previous rows within the same category.
    - shift() prevents the current row's target from leaking into its own encoding.

    For the validation fold:
    - Encoding values are learned only from the training fold.
    - Unseen categories are filled with the training fold global fraud rate.
    """

    # Sort each fold chronologically to ensure "past → future" order
    train_fold = train_fold.sort_values('trans_date_trans_time').copy()
    val_fold = val_fold.sort_values('trans_date_trans_time').copy()

    # Overall fraud rate in the training fold
    # Used as fallback for first occurrences or unseen categories
    global_mean = train_fold[target].mean()

    # For training rows:
    # group by the categorical column, e.g. merchant_clean
    # shift() removes the current row's target.
    # This ensures that only past information is used and prevents data leakage.
    # expanding().mean() computes historical fraud rate up to the previous row
    train_fold[f'{col}_te'] = (
        train_fold.groupby(col, observed=False)[target]
        .transform(lambda s: s.shift().expanding().mean())
    )

    # First occurrence of each category has no history, so fill with global mean
    train_fold[f'{col}_te'] = train_fold[f'{col}_te'].fillna(global_mean)

    # For validation rows:
    # learn category-level fraud rate from training fold only
    te_map = train_fold.groupby(col, observed=False)[target].mean()

    # Map validation category values to the training-learned fraud rate
    val_fold[f'{col}_te'] = val_fold[col].map(te_map)

    # If validation has a category never seen in training, fill with global mean
    val_fold[f'{col}_te'] = val_fold[f'{col}_te'].fillna(global_mean)

    return train_fold, val_fold

In [8]:
# ============================================================
# 1. Prepare time-sorted data
# ============================================================

def prepare_time_split_data(df, time_col='trans_date_trans_time'):
    """
    Convert the transaction time column to datetime and sort the dataset chronologically.
    
    This step is required before TimeSeriesSplit because the split depends on row order.
    """
    
    df = df.copy()
    
    # Convert transaction time to datetime format
    df[time_col] = pd.to_datetime(df[time_col])
    
    # Sort the full dataset by time so earlier rows represent earlier transactions
    df = df.sort_values(time_col).reset_index(drop=True)
    
    return df

In [9]:
# ============================================================
# 2. Prepare fold-level features
# ============================================================

def prepare_fold_features(train_fold, val_fold, feature_cols, target_col='is_fraud'):
    """
    Prepare model-ready X/y for one train-validation fold.
    
    This function performs fold-safe feature engineering:
    1. Time-safe target encoding for merchant_clean
    2. Binary encoding for gender
    3. One-hot encoding for category
    4. Missing indicator + median imputation
    5. Align train and validation feature columns
    """
    
    train_fold = train_fold.copy()
    val_fold = val_fold.copy()

    # --------------------------------------------------------
    # 1.1 Target encoding for merchant_clean
    # --------------------------------------------------------
    # Creates merchant_clean_te using only historical / train-fold information
    train_fold, val_fold = time_based_target_encode_train_val(
        train_fold,
        val_fold,
        col='merchant_clean',
        target=target_col
    )

    # # --------------------------------------------------------
    # # 1.2 Amount binning + target encoding
    # # --------------------------------------------------------
    # bins = [
    #     0, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, np.inf
    # ]

    # # Create categorical amt bins
    # train_fold['amt_bin'] = pd.cut(train_fold['amt'], bins=bins, include_lowest=True)
    # val_fold['amt_bin'] = pd.cut(val_fold['amt'], bins=bins, include_lowest=True)

    # # Time-aware target encoding for amount bins
    # train_fold, val_fold = time_based_target_encode_train_val(
    #     train_fold,
    #     val_fold,
    #     col='amt_bin',
    #     target=target_col
    # )

    # # Drop temporary amt_bin column
    # train_fold = train_fold.drop(columns=['amt_bin'])
    # val_fold = val_fold.drop(columns=['amt_bin'])

    # --------------------------------------------------------
    # 1.3 Quantile threshold - Hight amount flag
    # --------------------------------------------------------
    # Create amount threshold
    amt_threshold = train_fold['amt'].quantile(0.996)

    # Create high_amt_flag if transaction amount is bigger than amt_threshold
    train_fold['high_amt_flag'] = (train_fold['amt'] >= amt_threshold).astype(int)
    val_fold['high_amt_flag'] = (val_fold['amt'] >= amt_threshold).astype(int)

    # # --------------------------------------------------------
    # # 1.4 Amount percentile rank based on train fold only
    # # --------------------------------------------------------
    # # train fold fit percentile scale
    # # validation fold apply train percentile scale
    # train_amt_sorted = np.sort(train_fold['amt'].values)

    # train_fold['amt_rank_pct'] = (
    #     np.searchsorted(train_amt_sorted, train_fold['amt'], side='right')
    #     / len(train_amt_sorted)
    # )

    # val_fold['amt_rank_pct'] = (
    #     np.searchsorted(train_amt_sorted, val_fold['amt'], side='right')
    #     / len(train_amt_sorted)
    # )    
    
    # --------------------------------------------------------
    # 2. Gender binary encoding
    # --------------------------------------------------------
    # Convert gender from string to numeric feature
    train_fold['gender_bin'] = train_fold['gender'].map({'M': 0, 'F': 1})
    val_fold['gender_bin'] = val_fold['gender'].map({'M': 0, 'F': 1})

    # --------------------------------------------------------
    # 3. One-hot encoding for category
    # --------------------------------------------------------
    # One-hot encode category separately within train and validation folds
    train_cat = pd.get_dummies(train_fold['category'], prefix='category', dtype='int8')
    val_cat = pd.get_dummies(val_fold['category'], prefix='category', dtype='int8')

    # Use training fold categories as the feature space
    # If validation has unseen categories, they are dropped
    # If validation is missing train categories, they are filled with 0
    train_cat, val_cat = train_cat.align(
        val_cat,
        join='left',
        axis=1,
        fill_value=0
    )

    # Add one-hot encoded category columns back to train/validation folds
    train_fold = pd.concat([train_fold, train_cat], axis=1)
    val_fold = pd.concat([val_fold, val_cat], axis=1)

    # --------------------------------------------------------
    # 4. Define feature list
    # --------------------------------------------------------
    category_cols = train_cat.columns.tolist()
    
    current_features = (
        feature_cols
        + ['merchant_clean_te','high_amt_flag', 'gender_bin'] # 'amt_bin_te', 'amt_rank_pct'
        + category_cols
    )

    # --------------------------------------------------------
    # 5. Prepare X and y
    # --------------------------------------------------------
    X_train = train_fold[current_features].copy()
    y_train = train_fold[target_col].copy()

    X_val = val_fold[current_features].copy()
    y_val = val_fold[target_col].copy()

    # --------------------------------------------------------
    # 6. Missing indicators + median imputation
    # --------------------------------------------------------
    # These variables are missing for first transaction per cc_num because time_gap/speed require prior history
    impute_cols = ['txn_speed_kmh_clip_log', 'time_gap_log']

    for col in impute_cols:
        # Missing indicator captures whether the value was originally missing
        X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
        X_val[f'{col}_missing'] = X_val[col].isna().astype(int)

        # Median is learned from training fold only to avoid leakage
        median = X_train[col].median()
        X_train[col] = X_train[col].fillna(median)
        X_val[col] = X_val[col].fillna(median)

    # Add missing indicator columns into the final feature list
    current_features = current_features + [f'{col}_missing' for col in impute_cols]

    # --------------------------------------------------------
    # 7. Final train/validation feature alignment
    # --------------------------------------------------------
    # Keep the same feature order for train and validation
    X_train = X_train[current_features]
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

    return X_train, y_train, X_val, y_val

In [10]:
# ============================================================
# 3. Build model dictionary
# ============================================================

def build_models(pos_weight):
    """
    Build all candidate models for comparison.
    
    pos_weight is used by XGBoost to handle class imbalance:
    scale_pos_weight = negative samples / positive samples
    """
    
    models = {
        # ----------------------------------------------------
        # Baseline 1: Logistic Regression
        # ----------------------------------------------------
        # Logistic regression needs scaling because it is scale-sensitive
        'lr': Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(max_iter=1000))
        ]),

        # ----------------------------------------------------
        # Baseline 2: Random Forest
        # ----------------------------------------------------
        'rf': RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        ),

        # ----------------------------------------------------
        # Model 1: XGBoost
        # ----------------------------------------------------
        'xgb': XGBClassifier(
            n_estimators=100,
            scale_pos_weight=pos_weight,
            random_state=42,
            tree_method='hist',
            n_jobs=-1,
            learning_rate=0.05,
            eval_metric='logloss',
            subsample=0.8,
            colsample_bytree=0.8
        ),

        # ----------------------------------------------------
        # Model 2: LightGBM
        # ----------------------------------------------------
        'lightgbm': LGBMClassifier(
            max_depth=5,
            random_state=42,
            verbose=-1
        ),

        # ----------------------------------------------------
        # Model 3: CatBoost
        # ----------------------------------------------------
        'cat': CatBoostClassifier(
            iterations=200,
            learning_rate=0.05,
            depth=6,
            eval_metric='AUC',
            verbose=0,
            random_state=42
        )
    }
    
    return models

In [11]:
# ============================================================
# 4. Evaluate models on one fold
# ============================================================

def evaluate_models_on_fold(models, X_train, y_train, X_val, y_val, fold):
    """
    Train each model on the current training fold, predict on validation fold,
    calculate ROC-AUC and PR-AUC, and return fold-level results.
    """
    
    fold_results = []
    fold_rows = []

    for model_name, model in models.items():
        
        # Train current model
        model.fit(X_train, y_train)
        
        # Predict probability of positive class: fraud = 1
        y_prob = model.predict_proba(X_val)[:, 1]

        # Evaluate model performance
        roc = roc_auc_score(y_val, y_prob)
        pr = average_precision_score(y_val, y_prob)

        # Store full result for final summary
        fold_results.append({
            'fold': fold,
            'model': model_name,
            'roc_auc': roc,
            'pr_auc': pr,
            'train_size': len(X_train),
            'val_size': len(X_val),
            'fraud_rate_train': y_train.mean(),
            'fraud_rate_val': y_val.mean()
        })

        # Store compact result for fold-level display
        fold_rows.append({
            'model': model_name,
            'roc_auc': roc,
            'pr_auc': pr
        })

    # Create clean display table for this fold
    fold_display = (
        pd.DataFrame(fold_rows)
        .round(4)
        .sort_values('pr_auc', ascending=False)
        .reset_index(drop=True)
    )

    return fold_results, fold_display

In [12]:
# ============================================================
# Main function: Time-series cross-validation
# ============================================================

def run_time_series_cv(df, feature_cols, target_col='is_fraud', n_splits=5):
    """
    Run time-series cross-validation for multiple models.
    
    Workflow:
    1. Sort data chronologically
    2. Split data using TimeSeriesSplit
    3. Prepare fold-level features
    4. Build models
    5. Train/evaluate models on each fold
    6. Summarize performance by model
    """
    
    # Step 1: prepare chronological dataset
    df = prepare_time_split_data(df)

    # Step 2: initialize time-series CV
    tscv = TimeSeriesSplit(n_splits=n_splits)
    all_results = []

    # Step 3: loop through each time-based fold
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df), start=1):

        # Split into train and validation folds
        train_fold = df.iloc[train_idx].copy()
        val_fold = df.iloc[val_idx].copy()

        # Prepare model-ready features for the current fold
        X_train, y_train, X_val, y_val = prepare_fold_features(
            train_fold=train_fold,
            val_fold=val_fold,
            feature_cols=feature_cols,
            target_col=target_col
        )

        # Compute positive class weight for imbalanced fraud target
        pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

        # Build candidate models
        models = build_models(pos_weight)

        # Train and evaluate models on current fold
        fold_results, fold_display = evaluate_models_on_fold(
            models=models,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            fold=fold
        )

        # Add current fold results to global results list
        all_results.extend(fold_results)

        # Print fold-level model ranking
        print(f"\n{'='*70}")
        print(f"FOLD {fold}")
        print(f"{'='*70}")
        print(fold_display)

    # Convert all fold results into a dataframe
    results_df = pd.DataFrame(all_results)

    # Summarize model performance across folds
    summary = (
        results_df
        .groupby('model', as_index=False)
        .agg(
            mean_roc_auc=('roc_auc', 'mean'),
            std_roc_auc=('roc_auc', 'std'),
            mean_pr_auc=('pr_auc', 'mean'),
            std_pr_auc=('pr_auc', 'std')
        )
        .sort_values('mean_pr_auc', ascending=False)
    )

    # Print final summary
    print("\n" + "=" * 70)
    print("MEAN PERFORMANCE SUMMARY")
    print("=" * 70)
    print(summary.round(4))
    
    return results_df, summary

In [13]:
# # from catboost import CatBoostClassifier
# # from lightgbm import LGBMClassifier
# # from sklearn.linear_model import LogisticRegression
# # from sklearn.preprocessing import StandardScaler

# def run_time_series_cv(df, feature_cols, target_col='is_fraud', n_splits=5):

#     df = df.copy()

#     # Convert to datetime
#     df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    
#     # Global time sort
#     df = df.sort_values('trans_date_trans_time').reset_index(drop=True).copy()

#     tscv = TimeSeriesSplit(n_splits=n_splits)
#     fold_results = []

#     for fold, (train_idx, val_idx) in enumerate(tscv.split(df), start=1):
#         fold_rows = []
        
#         train_fold = df.iloc[train_idx].copy()
#         val_fold = df.iloc[val_idx].copy()

#         # ---------------------------------------
#         # 1. Target encoding (merchant only for now)
#         # ---------------------------------------
#         train_fold, val_fold = time_based_target_encode_train_val(
#             train_fold, val_fold, col='merchant_clean', target=target_col
#         )

#         # ---------------------------------------
#         # 2. One-hot encoding for gender & category
#         # ---------------------------------------

#         # a). Gender -> binary
#         train_fold['gender_bin'] = train_fold['gender'].map({'M':0, 'F':1})
#         val_fold['gender_bin'] = val_fold['gender'].map({'M':0, 'F':1})

#         # b). One-hot encode category inside each fold
#         train_cat = pd.get_dummies(train_fold[['category']], prefix='category', dtype='int8')
#         val_cat = pd.get_dummies(val_fold[['category']], prefix='category', dtype='int8')

#         train_cat, val_cat = train_cat.align(val_cat, join='left', axis=1, fill_value=0)

#         # c). Add One-hot encode category back to each fold
#         train_fold = pd.concat([train_fold, train_cat], axis=1)
#         val_fold = pd.concat([val_fold, val_cat], axis=1)
        
#         # ---------------------------------------
#         # 3. Define final feature list
#         # ---------------------------------------
#         category_cols = train_cat.columns.tolist()
#         current_features = feature_cols + ['merchant_clean_te', 'gender_bin'] + category_cols

#         # ---------------------------------------
#         # 4. Prepare X/y
#         # ---------------------------------------
#         X_train = train_fold[current_features].copy()
#         y_train = train_fold[target_col].copy()

#         X_val = val_fold[current_features].copy()
#         y_val = val_fold[target_col].copy()

#         # ---------------------------------------
#         # 5. Columns to impute: Missing indicator + median imputation
#         # ---------------------------------------
#         impute_cols = ['txn_speed_kmh_clip_log', 'time_gap_log']

#         for col in impute_cols:
#             X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
#             X_val[f'{col}_missing'] = X_val[col].isna().astype(int)
    
#             median = X_train[col].median()
#             X_train[col] = X_train[col].fillna(median)
#             X_val[col] = X_val[col].fillna(median)

#         # Add missing-indicators columns into feature list
#         current_features = current_features + [f'{col}_missing' for col in impute_cols]

#         # ---------------------------------------
#         # 6. Explicity align train/val columns
#         # ---------------------------------------

#         # X_train = X_train[current_features]
#         # X_val = X_val[current_features]
#         X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)

#         # ---------------------------------------
#         # 7. Positive class weight for imbalanced data
#         # ---------------------------------------
#         pos_weight = ((y_train==0).sum()) / ((y_train==1).sum()) # scale_pos_weight = neg/pos
        
#         # --->>> model.fit / predict / metrics

#         models = {
#             # ------------------------------------
#             # Baseline 1 - Logistic Regression
#             # ------------------------------------
#             'lr': Pipeline([
#                 ('scaler', StandardScaler()),
#                 ('lr', LogisticRegression(max_iter=1000))
#             ]),

#             # ------------------------------------
#             # Baseline 2 - Random Forest Modeling
#             # ------------------------------------
#             'rf': RandomForestClassifier(
#                 n_estimators=100,
#                 max_depth=None,
#                 random_state=42,
#                 n_jobs=1
#             ),
            
#             # ------------------------------------
#             # 8.1 Model 1: XGBoost
#             # ------------------------------------
#             'xgb': XGBClassifier(
#                 n_estimators=100,
#                 scale_pos_weight=pos_weight,
#                 random_state=42,
#                 tree_method='hist',
#                 n_jobs=-1,
#                 learning_rate=0.05,
#                 eval_metric='logloss', # Log loss measures how well the predicted probabilities match the true labels (not the final classification outcome)
#                 subsample=0.8, # subsampling from each tree
#                 colsample_bytree=0.8 # colsample_bytree: the fraction of features (columns) randomly sampled for each tree.
#             ),

#             # ------------------------------------
#             # 8.2 Model 2: LightBGM
#             # ------------------------------------
#             'lightbgm': LGBMClassifier(
#                 max_depth=5,
#                 random_state=42,
#                 verbose=-1
#             ),

#             # ------------------------------------
#             # 8.3 Model 3: Catboost
#             # ------------------------------------
#             'cat': CatBoostClassifier(
#                 iterations=200,
#                 learning_rate=0.05,
#                 depth=6,
#                 eval_metric='AUC',
#                 verbose=0,
#                 random_state=42
#             )
#         }

#         for model_name, model in models.items():

#             model.fit(X_train, y_train)
#             y_prob = model.predict_proba(X_val)[:, 1]

#             roc = roc_auc_score(y_val, y_prob)
#             pr = average_precision_score(y_val, y_prob)

#             # Save fold results
#             fold_results.append({
#                 'fold':fold,
#                 'model':model_name,
#                 'roc_auc':roc,
#                 'pr_auc':pr,
#                 'train_size':len(X_train),
#                 'val_size':len(X_val),
#                 'fraud_rate_train':y_train.mean(),
#                 'fraud_rate_val':y_val.mean()
#             })

#             fold_rows.append({
#                 'model': model_name,
#                 'roc_auc': roc,
#                 'pr_auc': pr
#             })
            
#             # print(f"[{model_name}] ROC: {roc:4f} | PR-AUC: {pr:.4f}")
#             # print("-" * 60)
#         print(f"\n{'='*70}")
#         print(f"FOLD {fold}")
#         print(f"{'='*70}")
#         print(pd.DataFrame(fold_rows).round(4).sort_values('pr_auc', ascending=False).reset_index(drop=True))


#         # # ------------------------------------
#         # # 8.1 Model 1: XGBoost
#         # # ------------------------------------
#         # fraud_xgb = XGBClassifier(
#         #     n_estimators=100,
#         #     scale_pos_weight=pos_weight,
#         #     random_state=42,
#         #     tree_method='hist',
#         #     n_jobs=-1,
#         #     learning_rate=0.05,
#         #     eval_metric='logloss', # Log loss measures how well the predicted probabilities match the true labels (not the final classification outcome)
#         #     subsample=0.8, # subsampling from each tree
#         #     colsample_bytree=0.8 # colsample_bytree: the fraction of features (columns) randomly sampled for each tree.
#         # )
    
#         # fraud_xgb.fit(X_train, y_train)
#         # y_prob_xgb = fraud_xgb.predict_proba(X_val)[:, 1]

#         # # ------------------------------------
#         # # 8.2 Model 2: Logistic Regression
#         # # ------------------------------------
#         # fraud_logistic = LogisticRegression(max_iter=1000, random_state=42)
#         # fraud_logistic.fit(X_train, y_train)
#         # y_prob_logistic = fraud_logistic.predict_proba(X_val)[:, 1]

#         # # ------------------------------------
#         # # 9. Evalution
#         # # ------------------------------------
#         # roc = roc_auc_score(y_val, y_prob_xgb)
#         # pr = average_precision_score(y_val, y_prob_xgb)

#         # # Save fold results
#         # fold_results.append({
#         #     'fold':fold,
#         #     'roc_auc':roc,
#         #     'pr_auc':pr,
#         #     'train_size':len(X_train),
#         #     'val_size':len(X_val),
#         #     'fraud_rate_train':y_train.mean(),
#         #     'fraud_rate_val':y_val.mean()
#         # })

#         # print(f"Fold {fold}")
#         # print(f"ROC-AUC: {roc:.4f}")
#         # print(f"PR-AUC: {pr:.4f}")
#         # print("-" * 60)

#     # Convert results to dataframe
#     results_df = pd.DataFrame(fold_results)

#     # print(f"\n{'='*70}")
#     # print("MEAN PERFORMANCE SUMMARY")
#     # print(f"{'='*70}")
#     # print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} ± {results_df['roc_auc'].std():.4f}")
#     # print(f"Mean PR-AUC: {results_df['pr_auc'].mean():.4f} ± {results_df['pr_auc'].std():.4f}")
#     summary = (
#         results_df
#         .groupby('model', as_index=False)
#         .agg(
#             mean_roc_auc=('roc_auc', 'mean'),
#             std_roc_auc=('roc_auc', 'std'),
#             mean_pr_auc=('pr_auc', 'mean'),
#             std_pr_auc=('pr_auc', 'std')
#         )
#         .sort_values('mean_pr_auc', ascending=False)
#     )
    
#     print("\n" + "=" * 70)
#     print("MEAN PERFORMANCE SUMMARY")
#     print("=" * 70)
#     print(summary.round(4))
    
#     return results_df, summary
#     # print(f"{'='*70}")

In [14]:
feature_cols = [
    # -----------------------------
    # Time features
    # -----------------------------
    "year", "month",
    "is_weekend", "is_night",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",

    # -----------------------------
    # Behavioral features
    # -----------------------------
    "txn_distance_km",
    "txn_speed_kmh_clip_log",
    "time_gap_log",

    # -----------------------------
    # Transaction amount
    # -----------------------------
    "amt_log",

    # -----------------------------
    # Demographics
    # -----------------------------
    "city_pop_log", "age"
]

In [15]:
run_time_series_cv(
    trainM, feature_cols, target_col='is_fraud', n_splits=5
)


FOLD 1
      model  roc_auc  pr_auc
0       cat   0.9954  0.8985
1  lightgbm   0.9933  0.8632
2       xgb   0.9957  0.8322
3        rf   0.9788  0.8222
4        lr   0.9090  0.3942

FOLD 2
      model  roc_auc  pr_auc
0       cat   0.9942  0.8884
1  lightgbm   0.9895  0.8351
2        rf   0.9734  0.8291
3       xgb   0.9967  0.8143
4        lr   0.8999  0.3865

FOLD 3
      model  roc_auc  pr_auc
0       cat   0.9953  0.9025
1        rf   0.9809  0.8578
2  lightgbm   0.9788  0.8457
3       xgb   0.9976  0.8364
4        lr   0.9194  0.4688

FOLD 4
      model  roc_auc  pr_auc
0       cat   0.9956  0.9102
1  lightgbm   0.9927  0.9028
2        rf   0.9863  0.8798
3       xgb   0.9977  0.8600
4        lr   0.9164  0.4809

FOLD 5
      model  roc_auc  pr_auc
0       cat   0.9968  0.9218
1        rf   0.9849  0.8889
2       xgb   0.9982  0.8522
3  lightgbm   0.9784  0.8402
4        lr   0.9151  0.4606

MEAN PERFORMANCE SUMMARY
      model  mean_roc_auc  std_roc_auc  mean_pr_auc  std_pr_auc


(    fold     model   roc_auc    pr_auc  train_size  val_size  \
 0      1        lr  0.908976  0.394217      216115    216112   
 1      1        rf  0.978820  0.822246      216115    216112   
 2      1       xgb  0.995680  0.832155      216115    216112   
 3      1  lightgbm  0.993287  0.863249      216115    216112   
 4      1       cat  0.995407  0.898483      216115    216112   
 5      2        lr  0.899922  0.386543      432227    216112   
 6      2        rf  0.973425  0.829079      432227    216112   
 7      2       xgb  0.996714  0.814277      432227    216112   
 8      2  lightgbm  0.989524  0.835137      432227    216112   
 9      2       cat  0.994162  0.888367      432227    216112   
 10     3        lr  0.919362  0.468797      648339    216112   
 11     3        rf  0.980901  0.857838      648339    216112   
 12     3       xgb  0.997643  0.836441      648339    216112   
 13     3  lightgbm  0.978839  0.845724      648339    216112   
 14     3       cat  0.99

***CatBoost*** achieved the best overall performance with the ***highest mean PR-AUC***, followed by ***Random Forest*** and ***XGBoost***.

***LightGBM*** showed relatively unstable performance across folds, while ***Logistic Regression*** served as a useful ***baseline model*** but underperformed compared with tree-based ensemble models.

# Comparison: Model Performance Across Amount Engineering Strategies

| Model | Baseline PR-AUC | `amt_bin_te` | `high_amt_flag` | `amt_rank_pct` |
|---|---:|---:|---:|---:|
| CatBoost | 0.9025 | 0.9038 | **0.9043** | 0.9015 |
| Random Forest | 0.8624 | 0.8677 | 0.8556 | **0.8802** |
| XGBoost | 0.8583 | 0.8384 | 0.8390 | 0.8394 |
| LightGBM | 0.8124 | 0.7115 | **0.8574** | 0.8562 |
| Logistic Regression | 0.4617 | **0.4710** | 0.4382 | 0.4214 |

---

# Feature Engineering Strategies Compared

## 1. Baseline

The baseline model used:

```python
amt_log
+ merchant_clean_te
+ category features
+ behavioral features
```

without any additional amount-based engineered features.

---

## 2. `amt_bin_te`

This approach:

- manually binned transaction amounts into predefined ranges
- applied time-aware target encoding to amount buckets
- generated the feature:

```python
amt_bin_te
```

This strategy introduced additional complexity and temporal target encoding variance.

---

## 3. `high_amt_flag`

This approach created a binary tail-risk indicator:

```python
amt_threshold = train_fold['amt'].quantile(0.996)

train_fold['high_amt_flag'] = (
    train_fold['amt'] >= amt_threshold
).astype(int)
```

The feature simply indicates whether a transaction belongs to the extreme upper tail of the transaction amount distribution.

This strategy produced:

- the best overall CatBoost PR-AUC
- substantially improved LightGBM stability
- simpler and more interpretable behavior than `amt_bin_te`

---

## 4. `amt_rank_pct`

This approach transformed transaction amount into a percentile-based continuous feature:

```python
train_fold['amt_rank_pct']
```

which represents the relative ranking of a transaction amount within the training fold distribution.

Unlike `high_amt_flag`, this method preserves continuous percentile information rather than using a binary threshold.

---

# Key Findings

- **CatBoost** remained the strongest overall model across all experiments.

- Adding `amt_bin_te` produced only marginal gains for CatBoost while introducing instability for XGBoost and LightGBM.

- `high_amt_flag` produced the best CatBoost performance overall and significantly improved LightGBM compared with `amt_bin_te`.

- `amt_rank_pct` did not improve CatBoost performance and slightly reduced PR-AUC relative to the baseline.

- **Random Forest** benefited the most from `amt_rank_pct`, achieving the highest Random Forest PR-AUC among all feature engineering strategies.

- **XGBoost** consistently underperformed after adding additional amount-engineering features, suggesting that the new amount-derived variables may be redundant or introduce noisy splits.

- **Logistic Regression** showed modest improvement with `amt_bin_te`, but performance deteriorated with both `high_amt_flag` and `amt_rank_pct`.

---

# Interpretation

The experiments suggest that fraud behavior is more strongly associated with:

```text
extreme-tail transaction behavior
```

rather than:

```text
fine-grained amount segmentation
```

or:

```text
continuous percentile ranking
```

In particular:

- `high_amt_flag` effectively captured large anomalous transactions with minimal feature engineering complexity.

- `amt_bin_te` introduced additional temporal target encoding noise and did not consistently improve model performance.

- `amt_rank_pct` preserved relative ordering information but did not provide meaningful gains for CatBoost.

---

# Conclusion

At this stage:

- `amt_bin_te` does not appear to provide sufficient benefit relative to its complexity.

- `amt_rank_pct` provides limited value for CatBoost-based fraud modeling.

- `high_amt_flag` currently represents the most effective and stable amount-engineering strategy.

---

# Current Recommended Feature Configuration

```python
amt_log
+ high_amt_flag
+ merchant_clean_te
+ category features
+ behavioral features
+ demographic features
```

This configuration currently provides the best balance of:

- predictive performance
- stability
- interpretability
- implementation simplicity